<h1> ADVANCE RAG </H1>


<h1>DOCUMENT LOADER</h1>

In [24]:
from langchain_community.document_loaders import Docx2txtLoader
import os 
def document_loader(folder_path):

    document = []

    for filename in os.listdir(folder_path):
        if filename.endswith('.docx'):
            file_path = os.path.join(folder_path,filename)

            loader = Docx2txtLoader(file_path)
            pages = loader.load()

            document.extend(pages)
    return document






<h1>TEXT SPLITTER</h1>

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_splitter(document):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 600,
        chunk_overlap = 60
    )

    chunks = splitter.split_documents(document)

    return chunks

<h1>EMBEDDING AND VECTOR DB</h1>

In [4]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def create_vector_db(chunks):

    embeding = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-v2')

    vector_db = FAISS.from_documents(
        chunks,
        embeding
    )

    return vector_db

<h1>KEYWORD SEARCHER</h1>

In [32]:
from rank_bm25 import BM25Okapi

class KeyWord_Search:
    def __init__(self,chunks):
        self.chunks = chunks

        tokenized_chunk = []

        for c in chunks:
            tokens = c.page_content.lower().split()
            tokenized_chunk.append(tokens)

        self.bm250 = BM25Okapi(tokenized_chunk)

    def search(self,query,k=5):

        question = query.lower().split()

        scores = self.bm250.get_scores(question)

        ranked_answers = sorted(range(len(scores)),
                                key = lambda index:scores[index],
                                reverse=True)
        top_answer = ranked_answers[:k]

        result = []

        for x in top_answer:
            result.append(self.chunks[x])
        return result

<h1>HYBRID SEARCH</h1>

In [10]:
class Hybrid_Search:
    def __init__(self,vector_db,bm250_db):
        self.vector_db = vector_db
        self.bm250_db = bm250_db

    def combined_search(self,query,k=5):
        vector_result = self.vector_db.similarity_search(query,k=k)
        bm250_result = self.bm250_db.search(query,k=k)

        combine_result = vector_result + bm250_result

        unique = []
        seen_content = set()

        for x in combine_result:
            if x.page_content not in seen_content:
                unique.append(x)
                seen_content.add(x.page_content)
        return unique


<h1>RERANKER</h1>

In [43]:
from sentence_transformers import CrossEncoder

class ReRanker:
    def __init__(self):
        self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rerank(self,query,document,k=5):

        pairs = []

        for c in document:
            pairs.append([
                query,
                c.page_content
            ])

        scores = self.model.predict(pairs)

        ranked_result = sorted(zip(scores,document),
                               key = lambda i:i[0],
                               reverse = True)

        top_answers = ranked_result[:k]

        result = []

        for scores,top in top_answers:
            result.append(top)
        return result


<h1>Prompt</h1>

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

def generate_answers(question,document):

    context = ""

    for c in document:
        context += c.page_content
        context += "\n\n"

    prompt = ChatPromptTemplate.from_template(
        '''
        You have to answer query from the given context. 

        if the question is out of context. just say,"QUESTION OUT OF CONTEXT"
        
        context
        {context}

        question
        {question}
'''
    )

    llm_model = ChatGroq(model='openai/gpt-oss-20b',
                         temperature = 0)

    chain = prompt | llm_model

    response = chain.invoke({
        'context':context,
        'question':question
    })


    return response.content

<h1>PIPELINE</h1>

In [45]:
import gradio as gr


# document load
folder_path = 'E:\GEN-AI-PROJECTS'
document = document_loader(folder_path)

# text splitter
chunks = text_splitter(document)

# create vector db
vector_db = create_vector_db(chunks)

#create keyword search
keywordSearch = KeyWord_Search(chunks)

# create hybrid searcher
hybrisSearch = Hybrid_Search(vector_db,keywordSearch)

#ranker
ranker = ReRanker()

def generate_final_answer(query,k=5):

    retived_documents = hybrisSearch.combined_search(query,k=k)

    reranked_final = ranker.rerank(query,retived_documents,k=k)

    answer = generate_answers(query,reranked_final)

    sources = ""

    for i,doc in enumerate(reranked_final):
        file_source = doc.metadata.get(
            'page',
            'unknown'
        )
        sources += f'\nSource {i+1}: Page {file_source}'
    final_response = answer
    final_response+='\n\nSources'
    final_response += sources

    return final_response


demo = gr.Interface(
        fn = generate_final_answer,
        inputs = gr.Textbox(
            label = 'Enter your question'
        ),
        outputs = gr.Textbox(
            label = 'Answer: ',
            lines = 15
        ),
        title = 'DEEP KNOWLEDGE '
    )

demo.launch(share=True)



    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7865

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
